# **ColabAlign**

## Fast pairwise protein secondary structure comparisons using multiprocessing

 [![Code DOI](https://img.shields.io/badge/DOI-10.5281/zenodo.14169501-blue)](https://doi.org/10.5281/zenodo.14169501) [![Paper DOI](https://img.shields.io/badge/DOI-10.1101/2025.10.06.677802-green)](https://doi.org/10.1101/2025.10.06.677802)

---

Create a phylogenetic tree that compares the secondary structure of proteins, rather than nucleotide or amino acid sequence. Scoring uses the [US-align](https://zhanggroup.org/US-align/) algorithm by [Zhang _et al._, (2022)](https://doi.org/10.1038/s41592-022-01585-1).

A score of **<0.17** indicates similarity indistinguishable from a random pair of structures, where as as score **≥0.50** indicates a pair with broadly the same fold ([Xu _et al._, 2010](https://doi.org/10.1093/bioinformatics/btq066))

---

## **Usage**

1) **Click `Runtime` -> `Change runtime type` -> `v5e-1 TPU`**

2) **Either upload `.pdb` and/or `.cif` format files directly or compressed into a `.zip` or `.tar.gz` file to the Colab instance by clicking the folder icon on the left, then dragging and dropping your structures.**

3) **Adjust the clustering threshold (horizontal bar) as needed.**

4) **Click `Run all`**

## N.B.

* Other runtimes can be used if you are paying for compute units. Ensure you select a runtime with more than 2 CPU cores, particularly if you plan to analyse a large dataset.

* Structure files containing more than one chain are split into separate .pdb files by BeEM prior to alignment. By default, only the **first chain** is considered (*first chain only* mode). If you want to use all chains, change `mode` to *all chains*

* Structural dendrograms generated by ColabAlign should be analysed as unrooted trees.

* Clustering `threshold` is applied to distances in the UPGMA tree by TreeCluster's default `max_clade` method. It is not a hard minimum pairwise TM-score cutoff.


In [ ]:
# @title Set up ColabAlign conda environment and prepare file structure

%%bash
set -euo pipefail

# ColabAlign v1.2.0 - commit 158663d
COLABALIGN_SHA="158663d8c246747aa019ab3323abf10a39ee074c"

rm -rf sample_data ColabAlign_git
git init -q ColabAlign_git
git -C ColabAlign_git fetch -q --depth 1 https://github.com/crfield18/ColabAlign.git "$COLABALIGN_SHA"
git -c advice.detachedHead=false -C ColabAlign_git checkout -q FETCH_HEAD


test "$(git -C ColabAlign_git rev-parse HEAD)" = "$COLABALIGN_SHA"

pip install konda==0.1.0
konda config --file /usr/local/.condarc --remove channels defaults
konda config --file /usr/local/.condarc --add channels conda-forge

konda create --yes -n colabalign --file ColabAlign_git/locks/conda-linux-64.lock --quiet
konda activate colabalign
konda run "python -m pip install --no-deps --no-build-isolation --quiet ./ColabAlign_git"
rm -rf ColabAlign_git


In [ ]:
# @title Align Structures
# Run colabalign with all available cores

threshold = 0.20 # @param {type:"slider", min:0.01, max:1.00, step:0.01}
mode = 'first chain only' # @param ["first chain only", "all chains"]

if mode == 'all chains':
    mode = 'all'
elif mode == 'first chain only':
    mode = 'first'

!mkdir -p results models
# Skip ./results (previous runs) so old outputs are never re-used as inputs.
# .zip files are moved into models/ as-is; colabalign extracts them.
!find . -name "*.tar.gz" -not -path "./results/*" -exec tar -xzf {} \; > /dev/null 2>&1
!find . -type f \( -iname "*.pdb" -o -iname "*.cif" -o -iname "*.zip" \) -not -name "colabalign_results_*.zip" -not -path "./models/*" -not -path "./results/*" | xargs -I {} mv {} models/ 2>/dev/null || true
!find ./models/ -name "._*" -delete 2>/dev/null

!echo "Starting alignment with $(nproc --all) cores..."
!konda run "python -u -m colabalign -i /content/models -o /content/results -c $(nproc --all) -t $threshold -m $mode"


In [ ]:
# @title Zip and download results

import os
import datetime
from google.colab import files

# Name the zipped results file using the current date and time
# to not accidentally overwrite older results files when downloading
current_dt = datetime.datetime.now()
zip_filename = f'colabalign_results_{current_dt.strftime("%Y%m%d-%H%M")}.zip'

# Using the built-in zip function rather than a python module for improved efficiency
os.system(f'zip -r {zip_filename} results -x "*.pkl"')
files.download(zip_filename)
